# Fase 4: integracion y visualizacion
**Grupo 10, UNAB.** Los registros de esta ejecucion son **sinteticos** y no corresponden a MetroPT-3 ni a eventos reales de BHP. Requiere ejecutar desde la raiz del proyecto.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, timeit
p=Path("datos/registros_limpios.csv")
df=pd.read_csv(p,parse_dates=["Fecha"])
assert len(df)>0
print("Duraciones ausentes:",int(df["Tiempo_reparacion_h"].isna().sum()))
print("Registros aceptados:",len(df)); display(df.head())

In [ ]:
conteos=df.Tecnologia.value_counts()
ax=conteos.plot.bar(title="Figura 1. Registros por tecnologia",ylabel="Numero de registros")
ax.set_xlabel("Tecnologia");plt.tight_layout();plt.savefig("figuras/F4_figura1_tecnologias.png",dpi=170);plt.show()

In [ ]:
res=df.groupby("Tecnologia").Tiempo_reparacion_h.agg(["mean","median"])
ax=res.plot.bar(title="Figura 2. Tiempo de reparacion",ylabel="Horas por registro")
ax.set_xlabel("Tecnologia");plt.tight_layout();plt.savefig("figuras/F4_figura2_tiempos.png",dpi=170);plt.show()

**Tratamiento de valores ausentes:** se conservan los registros aceptados para el conteo por tecnología. Para medianas y benchmarks de duración se excluyen únicamente los valores faltantes de esa variable; no se imputan tiempos no observados.

In [ ]:
a=df.Tiempo_reparacion_h.dropna().to_numpy();u=float(np.median(a))
def iterativo(x,u):return sum(1 for v in x if v>u)
def recursivo(x,u):
 if len(x)==0:return 0
 if len(x)==1:return int(x[0]>u)
 m=len(x)//2;return recursivo(x[:m],u)+recursivo(x[m:],u)
def vectorizado(x,u):return int(np.count_nonzero(x>u))
filas=[]
for n in [20,50,100,len(a)]:
 x=a[:n];assert iterativo(x,u)==recursivo(x,u)==vectorizado(x,u)
 for nombre,fn in [("Iterativo",iterativo),("Recursivo",recursivo),("Vectorizado",vectorizado)]:
  t=timeit.repeat(lambda:fn(x,u),number=20,repeat=7)
  filas.append(dict(n=n,algoritmo=nombre,microsegundos=float(np.median(t)/20*1e6)))
b=pd.DataFrame(filas);display(b)
b.to_csv("F4/benchmark_f4.csv",index=False)

In [ ]:
for nombre,g in b.groupby("algoritmo"):
 plt.plot(g.n,g.microsegundos,marker="o",label=nombre)
plt.title("Figura 3. Benchmark timeit");plt.xlabel("Registros (n)");plt.ylabel("Microsegundos por llamada");plt.legend();plt.tight_layout();plt.savefig("figuras/F4_figura3_benchmark.png",dpi=170);plt.show()

## Interpretacion
Los graficos 1 y 2 describen exclusivamente un conjunto **sintetico**. El grafico 3 mide tiempos en el equipo de ejecucion y no implica que las diferencias sean universales. La recursividad divide el arreglo y mantiene profundidad logaritmica. El percentil o mediana utilizados no representan umbrales de seguridad. Para el estudio MetroPT-3 se requiere obtener y ejecutar el CSV original por separado.